# チートによるコード改ざんを防ごう
---

#### プログラムを不正改造等のチート行為は著作者人格権侵害や電子計算機損壊等業務妨害罪などの法律に抵触する犯罪行為となりえます。このコードは教育目的であり、防御用ですが、絶対に本講義で得た知識を悪用しないでください。

---
## ・セクション1. 乱数

In [39]:
import random
import time
import hashlib
import sys

In [40]:
# --- 被害者側の処理 ---
print("--- 被害者側（乱数を使用する側） ---")
#シード値を用いることで、同じ乱数を作成できる
seed = 12345 
random.seed(seed)
first_number = random.randint(1, 1000)
secret_number = random.randint(1, 1000)
print(f"最初に生成した乱数: {first_number}")
print(f"次に生成する乱数: {secret_number}\n")

--- 被害者側（乱数を使用する側） ---
最初に生成した乱数: 427
次に生成する乱数: 751



In [41]:
# --- 攻撃者側の処理 ---
print("--- 攻撃者側（乱数を予測する側） ---")
known_seed = seed 
random.seed(known_seed)
dummy_number = random.randint(1, 1000)
predicted_number = random.randint(1, 1000)
print(f"攻撃者が再現したダミー乱数: {dummy_number}")
print(f"攻撃者が予測した次の乱数: {predicted_number}\n")

--- 攻撃者側（乱数を予測する側） ---
攻撃者が再現したダミー乱数: 427
攻撃者が予測した次の乱数: 751



In [42]:
#シード値がわかってしまうと危険 ->　時間をシード値にしてしまうとバレやすい
#演習１空欄を埋めて乱数を作成しよう

#時刻ベースのシードで乱数を生成する関数
def generate():
    #時刻シード
    seed = int(time.time())
    random.seed(seed) #あとで空欄
    outputs = [random.getrandbits(32) for _ in range(3)] #3回生成
    print("===脆弱性のある乱数生成 ===")
    print("シード値:　", seed)
    print("結果:", outputs)
    print()
    return seed, outputs

#seed値を解析する方法
def find_seed(outputs, time, window=600):
    start = time - window//2
    end   = time + window//2
    for find in range(start, end + 1):
        random.seed(find) #あとで空欄
        cand = [random.getrandbits(32) for _ in range(len(outputs))]
        if cand == outputs:
            print("解析したシード値: ", find)
            return find
    print("シード値の探索に失敗")
    return None

#生成される乱数を予測する方法
def predict(find, outputs):
    random.seed(find)
    _ = [random.getrandbits(32) for _ in range(len(outputs))]  # 既に観測された分を消費
    next_vals = [random.getrandbits(32) for _ in range(5)]
    print("次に生成される乱数の予測値:", next_vals)
    return next_vals

In [43]:
#それぞれの関数を使って実験
seed, outputs = generate()
time.sleep(2) #観測と攻撃に時間差がある想定
now_time = int(time.time())  #現在時刻
re_seed = find_seed(outputs, now_time, window=1200)
if re_seed is not None:
    predict(re_seed, outputs)
else:
    print("指定時間内にseed値の発見はできなかった")

===脆弱性のある乱数生成 ===
シード値:　 1764738752
結果: [2740251455, 3383322158, 1511377255]

解析したシード値:  1764738752
次に生成される乱数の予測値: [3637523271, 1419724239, 2306217394, 959389270, 84420959]


---
## ・セクション2 逆アセンブリング・メモリ

In [44]:
import dis

#この動作を逆アセンブルしてみる
def function(x, y):
    s = x * 2
    t = y + 3
    return s + t

print("=== SOURCE CODE ===")
print(function.__code__.co_consts)

print("\n=== DISASSEMBLED BYTECODE ===\n")
dis.dis(function)
print("\n==============================")

=== SOURCE CODE ===
(None, 2, 3)

=== DISASSEMBLED BYTECODE ===

  4           RESUME                   0

  5           LOAD_FAST                0 (x)
              LOAD_CONST               1 (2)
              BINARY_OP                5 (*)
              STORE_FAST               2 (s)

  6           LOAD_FAST                1 (y)
              LOAD_CONST               2 (3)
              BINARY_OP                0 (+)
              STORE_FAST               3 (t)

  7           LOAD_FAST_LOAD_FAST     35 (s, t)
              BINARY_OP                0 (+)
              RETURN_VALUE



In [45]:
#メモリ
import ctypes

x = 123
addr = id(x)
ptr = ctypes.cast(addr, ctypes.POINTER(ctypes.c_ulong))

#メモリを見てみる
print("保存場所: ", ptr)
print("値: ", ptr.contents)

保存場所:  <__main__.LP_c_ulong object at 0x1059683d0>
値:  c_ulong(4294967295)


In [46]:
#メモリの値書き換えを行なってみる
new_value = 0x1234567890ABCDEF
new_ulong = ctypes.c_ulong(new_value)
ptr.contents = new_ulong

## ・セクション3: 改ざんを検知し、チーターを追放しよう
---

In [38]:
#確認
ptr.contents

c_ulong(1311768467294899695)

In [64]:
# -----------------------------
# 監視対象
# -----------------------------
def target():
    return 42

buf = bytearray(b"TEST")

# オリジナル状態を記録
orig_code_hash = hashlib.sha256(target.__code__.co_code).hexdigest()
orig_buf_hash = hashlib.sha256(buf).hexdigest()

# -----------------------------
# チェック関数
# -----------------------------
def detect():
    alerts = []
    
    # バイトコード改ざん検知
    now_code_hash = hashlib.sha256(target.__code__.co_code).hexdigest()

    #if文の条件式を埋めよう
    if now_code_hash != orig_code_hash:
        alerts.append("Bytecode tampering detected!")
    
    # メモリ改ざん検知
    now_buf_hash = hashlib.sha256(buf).hexdigest()
    if now_buf_hash != orig_buf_hash:
        alerts.append("Memory tampering detected!")
    
    return alerts or ["No tampering detected."]

# -----------------------------
# テスト（改ざんを実際に行う）
# -----------------------------
def test():
    print("=== 1. 正常時 ===")
    print(detect())

    # -----------------------------------------
    # (1) バイトコード改ざん（code.replace を使用）
    # -----------------------------------------
    print("\n=== 2. バイトコード改ざん ===")
    hacked = target.__code__.replace(
        co_code=b'\x64\x01\x00\x53',   # LOAD_CONST 1, RETURN_VALUE
        co_consts=target.__code__.co_consts + (999,)
    )
    target.__code__ = hacked
    print(detect())

    # -----------------------------------------
    # (2) メモリ改ざん（bytearray の中身を書き換え）
    # -----------------------------------------
    print("\n=== 3. メモリ改ざん ===")
    ptr = (ctypes.c_char * len(buf)).from_address(id(buf) + 0x20)
    ptr[0] = b'X'[0]

    print(detect())

In [66]:
test()

=== 1. 正常時 ===
['No tampering detected.']

=== 2. バイトコード改ざん ===
['Bytecode tampering detected!']

=== 3. メモリ改ざん ===
['Bytecode tampering detected!']


In [67]:
#リバースエンジニアリングの検知
def check_debugger():
    for mod in sys.modules:
        if mod == "dis":
            print("Reverse-engineering tool detected!")
            return
    print("Safe")

check_debugger()

Reverse-engineering tool detected!
